# GOOD external-validity screening wave

Execution is disabled by default. Set EXECUTE=True only after all prerequisites and upstream provider/licence gates pass.

This notebook never downloads provider data and never treats two GPUs as data-parallel unless a runner explicitly declares that support.


In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys
REPO = Path('/kaggle/working/gnn-fraud') if Path('/kaggle').exists() else Path.cwd()
EXECUTE = False
WAVE = 'good'
MATRIX = REPO / 'configs/coregraph/run_matrices/GOOD_GRID.csv'
print({'repo': str(REPO), 'wave': WAVE, 'execute': EXECUTE})


In [ ]:
required = [REPO / 'coregraph', MATRIX, REPO / 'results/coregraph_build/ANALYSIS_PLAN_FREEZE.json']
missing = [str(path) for path in required if not path.exists()]
assert not missing, f'Missing prerequisites: {missing}'
rows = list(csv.DictReader(MATRIX.open()))
print({'matrix_rows': len(rows), 'matrix_sha256': hashlib.sha256(MATRIX.read_bytes()).hexdigest()})


In [ ]:
selected = [row for row in rows if row['execution_status'] == 'PLANNED']
print({'planned_rows': len(selected), 'blocked_rows': len(rows)-len(selected)})
assert all(row['runtime_status'] == 'TBD_PROFILE' for row in selected)
# Deterministic sharding across two T4 scheduling lanes; this is not DDP.
lanes = {0: selected[0::2], 1: selected[1::2]}
print({f't4_lane_{lane}': len(items) for lane, items in lanes.items()})


In [ ]:
if EXECUTE:
    raise RuntimeError('Heavy execution entry point intentionally requires the runbook command and staged manifests; do not toggle ad hoc.')
print('DRY_RUN_VALIDATED')
